Celda 1 - Importaciones

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

Celda 2 - Cargar los datasets

In [0]:
from pyspark.sql.functions import *

df_empresas = spark.read.format("delta").load("/Volumes/workspace/default/churn_b2b/Bronze/empresas")

df_facturacion = spark.read.format("delta").load("/Volumes/workspace/default/churn_b2b/Bronze/facturacion")

df_consumo = spark.read.format("delta").load("/Volumes/workspace/default/churn_b2b/Bronze/consumo")

df_servicios = spark.read.format("delta").load("/Volumes/workspace/default/churn_b2b/Bronze/servicios")

df_tickets = spark.read.format("delta").load("/Volumes/workspace/default/churn_b2b/Bronze/tickets")

df_eventos = spark.read.format("delta").load("/Volumes/workspace/default/churn_b2b/Bronze/eventos_red")

df_churn = spark.read.format("delta").load("/Volumes/workspace/default/churn_b2b/Bronze/churn")

print("Datos cargados desde Bronze correctamente")

Datos cargados desde Bronze correctamente


Celda 3 - Eliminar duplicados

In [0]:
df_empresas = df_empresas.dropDuplicates()
df_facturacion = df_facturacion.dropDuplicates()
df_consumo = df_consumo.dropDuplicates()
df_servicios = df_servicios.dropDuplicates()
df_tickets = df_tickets.dropDuplicates()
df_eventos = df_eventos.dropDuplicates()
df_churn = df_churn.dropDuplicates()

Celda 4 - Validar valores nulos

In [0]:
datasets = {
    "Empresas": df_empresas,
    "Facturación": df_facturacion,
    "Consumo": df_consumo,
    "Servicios": df_servicios,
    "Tickets": df_tickets,
    "Eventos": df_eventos,
    "Churn": df_churn
}

for nombre, df in datasets.items():
    print(f"\n===== {nombre} =====")

    df.select([
        count(when(col(c).isNull(), c)).alias(c)
        for c in df.columns
    ]).show()


===== Empresas =====
+----------+---+------------+------+------+---------+--------+----------+-------------------+---+----+
|company_id|ruc|razon_social|sector|region|empleados|segmento|fecha_alta|ejecutivo_comercial|nps|csat|
+----------+---+------------+------+------+---------+--------+----------+-------------------+---+----+
|         0|  0|           0|     0|     0|        0|       0|         0|                  0|  0|   0|
+----------+---+------------+------+------+---------+--------+----------+-------------------+---+----+


===== Facturación =====
+----------+----------+-------+---------------+---------+------------+
|factura_id|company_id|periodo|monto_facturado|dias_mora|deuda_actual|
+----------+----------+-------+---------------+---------+------------+
|         0|         0|      0|              0|        0|           0|
+----------+----------+-------+---------------+---------+------------+


===== Consumo =====
+----------+----------+-------+----------+------------------

Celda 5 - Verificar tipos de datos

In [0]:
for nombre, df in datasets.items():
    print(f"\n===== {nombre} =====")
    df.printSchema()


===== Empresas =====
root
 |-- company_id: integer (nullable = true)
 |-- ruc: long (nullable = true)
 |-- razon_social: string (nullable = true)
 |-- sector: string (nullable = true)
 |-- region: string (nullable = true)
 |-- empleados: integer (nullable = true)
 |-- segmento: string (nullable = true)
 |-- fecha_alta: date (nullable = true)
 |-- ejecutivo_comercial: string (nullable = true)
 |-- nps: integer (nullable = true)
 |-- csat: double (nullable = true)


===== Facturación =====
root
 |-- factura_id: integer (nullable = true)
 |-- company_id: integer (nullable = true)
 |-- periodo: integer (nullable = true)
 |-- monto_facturado: double (nullable = true)
 |-- dias_mora: integer (nullable = true)
 |-- deuda_actual: double (nullable = true)


===== Consumo =====
root
 |-- consumo_id: integer (nullable = true)
 |-- company_id: integer (nullable = true)
 |-- periodo: integer (nullable = true)
 |-- trafico_gb: double (nullable = true)
 |-- ancho_banda_promedio: double (nullable = t

Celda 6 - Normalizar company_id

In [0]:
df_empresas = df_empresas.withColumn("company_id", col("company_id").cast("int"))
df_facturacion = df_facturacion.withColumn("company_id", col("company_id").cast("int"))
df_consumo = df_consumo.withColumn("company_id", col("company_id").cast("int"))
df_servicios = df_servicios.withColumn("company_id", col("company_id").cast("int"))
df_tickets = df_tickets.withColumn("company_id", col("company_id").cast("int"))
df_eventos = df_eventos.withColumn("company_id", col("company_id").cast("int"))
df_churn = df_churn.withColumn("company_id", col("company_id").cast("int"))

Celda 7 - Crear vistas temporales Silver

In [0]:
df_empresas.createOrReplaceTempView("silver_empresas")
df_facturacion.createOrReplaceTempView("silver_facturacion")
df_consumo.createOrReplaceTempView("silver_consumo")
df_servicios.createOrReplaceTempView("silver_servicios")
df_tickets.createOrReplaceTempView("silver_tickets")
df_eventos.createOrReplaceTempView("silver_eventos")
df_churn.createOrReplaceTempView("silver_churn")

Celda 8 - Verificar integridad por empresa

In [0]:
print("Empresas:", df_empresas.select("company_id").distinct().count())
print("Churn:", df_churn.select("company_id").distinct().count())
print("Facturación:", df_facturacion.select("company_id").distinct().count())
print("Consumo:", df_consumo.select("company_id").distinct().count())

Empresas: 5000
Churn: 5000
Facturación: 5000
Consumo: 5000


Guarda en silver

In [0]:
df_empresas.write.mode("overwrite").format("delta").save("/Volumes/workspace/default/churn_b2b/Silver/empresas")

df_facturacion.write.mode("overwrite").format("delta").save("/Volumes/workspace/default/churn_b2b/Silver/facturacion")

df_consumo.write.mode("overwrite").format("delta").save("/Volumes/workspace/default/churn_b2b/Silver/consumo")

df_servicios.write.mode("overwrite").format("delta").save("/Volumes/workspace/default/churn_b2b/Silver/servicios")

df_tickets.write.mode("overwrite").format("delta").save("/Volumes/workspace/default/churn_b2b/Silver/tickets")

df_eventos.write.mode("overwrite").format("delta").save("/Volumes/workspace/default/churn_b2b/Silver/eventos_red")

df_churn.write.mode("overwrite").format("delta").save("/Volumes/workspace/default/churn_b2b/Silver/churn")